# Canada Electricity Transition Intelligence

## Introduction
This analysis estimates near-term electricity capacity pathways by region and energy source in Canada using the available annual capacity records. The objective is to produce forecasts that are technically defensible and practical for planning discussions.

## Background
The dataset is compact and annual. That matters because short time histories increase overfitting risk when model complexity is not controlled. For that reason, the analysis emphasizes transparent diagnostics, time-aware validation, and model selection based on out-of-time performance.

## Main Aim
Develop a reliable forecasting workflow that selects the most suitable model for each region-source series and reports uncertainty in a decision-ready format.

## Objectives
1. Establish the historical behavior of the data across trend, variability, and source composition.
2. Build a focused model portfolio that reflects the observed behavior of the series.
3. Evaluate model performance with chronology-preserving validation.
4. Generate forecast paths and uncertainty intervals that can support planning and policy conversations.
5. Conclude with findings and recommendations that are directly supported by computed evidence.

## Research Questions
1. Which model is most reliable for the selected region-source series under out-of-time evaluation?
2. Do observed trend, volatility, and persistence patterns support using a mixed model portfolio?
3. Are the final outputs usable for planning decisions in terms of direction, scale, and uncertainty?

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

In [2]:
# Load and clean the capacity dataset.
DATA_PATH = Path("electricity-capacity-dataset.csv")
base = pd.read_csv(DATA_PATH)
base = base.dropna(subset=["Region", "Source", "Year", "Data"]).copy()
base["Region"] = base["Region"].astype(str).str.strip()
base["Source"] = base["Source"].astype(str).str.strip()
base["Year"] = base["Year"].astype(int)
base["Data"] = base["Data"].astype(float)
base = base.sort_values(["Region", "Source", "Year"]).reset_index(drop=True)

# Keep explicit views so regional comparisons never include the national aggregate.
region_only = base[base["Region"] != "Canada"].copy()
canada_only = base[base["Region"] == "Canada"].copy()

# Quick quality profile used to justify next analysis steps.
profile = pd.DataFrame([
    {
        "Rows": len(base),
        "Regions": base["Region"].nunique(),
        "Sources": base["Source"].nunique(),
        "YearMin": int(base["Year"].min()),
        "YearMax": int(base["Year"].max()),
        "MissingAfterClean": int(base[["Region", "Source", "Year", "Data"]].isna().sum().sum()),
    }
])
display(profile)
display(base.head())

,Rows,Regions,Sources,YearMin,YearMax,MissingAfterClean
0,1344,14,8,2005,2016,0


,Region,Source,Year,Data,Unit
0,AB,Biomass,2005,271.0,MW
1,AB,Biomass,2006,313.1,MW
2,AB,Biomass,2007,313.1,MW
3,AB,Biomass,2008,313.1,MW
4,AB,Biomass,2009,323.2,MW


## Data Profile and Analytical Direction
The profile confirms a short annual panel with complete core fields after cleaning. This directly shapes the analytical path:
- time order must be preserved during validation to prevent future leakage,
- model complexity must be controlled because sample size is limited,
- performance should be judged on out-of-time errors rather than in-sample fit.

These constraints motivate the modeling choices used below.

## Transition Structure Across Regions and Sources

Before moving into model selection, this section establishes the structural evidence used in the app as well: regional renewable share, the shift in Canada’s source composition over time, the national capacity trend, the renewable versus non-renewable split, and the latest source mix. Together, these views separate regional comparison from national aggregation and make the transition pattern easier to interpret.

In [8]:
latest_year = int(base["Year"].max())
start_year = int(base["Year"].min())

renewable_sources = ["Hydro", "Solar", "Wind", "Tidal", "Wave", "Biomass", "Geothermal"]

# 1) Regional 100% stacked mix (exclude national aggregate Canada).
regional_latest = (
    region_only[region_only["Year"] == latest_year]
    .assign(EnergyType=lambda d: np.where(d["Source"].isin(renewable_sources), "Renewable", "Non-Renewable"))
    .groupby(["Region", "EnergyType"], as_index=False)["Data"]
    .sum()
)
regional_tot = regional_latest.groupby("Region", as_index=False)["Data"].sum().rename(columns={"Data": "Tot"})
regional_mix = regional_latest.merge(regional_tot, on="Region", how="left")
regional_mix["SharePct"] = np.where(regional_mix["Tot"] > 0, 100.0 * regional_mix["Data"] / regional_mix["Tot"], 0.0)

order_regions = (
    regional_mix[regional_mix["EnergyType"] == "Renewable"]
    .sort_values("SharePct", ascending=False)["Region"]
    .tolist()
)

fig_regional_mix = px.bar(
    regional_mix,
    x="Region",
    y="SharePct",
    color="EnergyType",
    barmode="stack",
    category_orders={"Region": order_regions, "EnergyType": ["Renewable", "Non-Renewable"]},
    color_discrete_map={"Renewable": "#2563eb", "Non-Renewable": "#dc2626"},
    labels={"SharePct": "Share (%)", "EnergyType": "Energy type"},
    title=f"Regional Mix Snapshot (100% Stacked) - {latest_year}",
)
fig_regional_mix.update_yaxes(range=[0, 100])
fig_regional_mix.update_layout(height=430, legend_title_text="", xaxis_title=f"Region ({latest_year})")
fig_regional_mix.show()

# 2) Source shift in Canada from the first to the latest year.
a = canada_only[canada_only["Year"] == start_year].groupby("Source", as_index=False)["Data"].sum()
b = canada_only[canada_only["Year"] == latest_year].groupby("Source", as_index=False)["Data"].sum()
src = sorted(set(a["Source"]).union(set(b["Source"])))
a = a.set_index("Source").reindex(src, fill_value=0).reset_index()
b = b.set_index("Source").reindex(src, fill_value=0).reset_index()
ta = float(a["Data"].sum())
tb = float(b["Data"].sum())
comp = pd.DataFrame(
    {
        "Source": src,
        f"Share {start_year}": [100.0 * float(v) / ta if ta > 0 else 0.0 for v in a["Data"]],
        f"Share {latest_year}": [100.0 * float(v) / tb if tb > 0 else 0.0 for v in b["Data"]],
    }
)
source_shift_long = comp.melt(
    id_vars="Source",
    value_vars=[f"Share {start_year}", f"Share {latest_year}"],
    var_name="Year",
    value_name="SharePct",
)

fig_source_shift = px.bar(
    source_shift_long,
    x="SharePct",
    y="Source",
    color="Year",
    barmode="group",
    orientation="h",
    color_discrete_map={f"Share {start_year}": "#94a3b8", f"Share {latest_year}": "#2563eb"},
    labels={"SharePct": "Share (%)"},
    title=f"Source Shift: {start_year} vs {latest_year}",
)
fig_source_shift.update_layout(height=430, legend_title_text="")
fig_source_shift.show()

# 3) Canada total capacity trend.
can_total = canada_only.groupby("Year", as_index=False)["Data"].sum()
fig_can_total = px.line(
    can_total,
    x="Year",
    y="Data",
    markers=True,
    labels={"Data": "Capacity (MW)"},
    title="Canada Total Capacity Trend",
)
fig_can_total.update_traces(line={"width": 3, "color": "#0f172a"})
fig_can_total.update_layout(height=380)
fig_can_total.show()

# 4) Canada renewable vs non-renewable over time.
can_mix_time = (
    canada_only
    .assign(EnergyType=lambda d: np.where(d["Source"].isin(renewable_sources), "Renewable", "Non-Renewable"))
    .groupby(["Year", "EnergyType"], as_index=False)["Data"]
    .sum()
)
piv_mix = can_mix_time.pivot(index="Year", columns="EnergyType", values="Data").fillna(0.0)
if "Renewable" not in piv_mix.columns:
    piv_mix["Renewable"] = 0.0
if "Non-Renewable" not in piv_mix.columns:
    piv_mix["Non-Renewable"] = 0.0

mix_plot = piv_mix.reset_index().melt(
    id_vars="Year",
    value_vars=["Renewable", "Non-Renewable"],
    var_name="EnergyType",
    value_name="Capacity",
)

fig_mix_line = px.area(
    mix_plot,
    x="Year",
    y="Capacity",
    color="EnergyType",
    color_discrete_map={"Renewable": "#2563eb", "Non-Renewable": "#dc2626"},
    title="Renewable vs Non-Renewable Capacity",
    labels={"Capacity": "Capacity (MW)", "EnergyType": "Energy type"},
)
fig_mix_line.update_layout(height=400, legend_title_text="")
fig_mix_line.show()

# 5) Latest Canada source mix doughnut.
share_latest = (
    canada_only[canada_only["Year"] == latest_year]
    .groupby("Source", as_index=False)["Data"]
    .sum()
    .sort_values("Data", ascending=False)
)

fig_donut = px.pie(
    share_latest,
    names="Source",
    values="Data",
    hole=0.52,
    title="Canada Source Mix Donut",
)
fig_donut.update_traces(textposition="inside", textinfo="percent+label")
fig_donut.update_layout(height=430)
fig_donut.show()

In [4]:
def fit_ses_alpha(y_train: np.ndarray) -> float:
    if len(y_train) < 3:
        return 0.4
    alphas = [0.2, 0.35, 0.5, 0.65, 0.8]
    best_alpha = 0.4
    best_err = float("inf")
    for alpha in alphas:
        level = float(y_train[0])
        errs = []
        for i in range(1, len(y_train)):
            pred = level
            errs.append((float(y_train[i]) - pred) ** 2)
            level = alpha * float(y_train[i]) + (1.0 - alpha) * level
        mse = float(np.mean(errs)) if errs else float("inf")
        if mse < best_err:
            best_err = mse
            best_alpha = alpha
    return float(best_alpha)


def ses_next(y_train: np.ndarray, alpha: float) -> float:
    level = float(y_train[0])
    for i in range(1, len(y_train)):
        level = alpha * float(y_train[i]) + (1.0 - alpha) * level
    return float(level)


def drift_next(y_train: np.ndarray) -> float:
    if len(y_train) < 2:
        return float(y_train[-1])
    drift = (float(y_train[-1]) - float(y_train[0])) / max(1, len(y_train) - 1)
    return float(y_train[-1]) + drift


def fit_model(name: str, x_train: np.ndarray, y_train: np.ndarray):
    if name == "NaiveLag1":
        return None, {}
    if name == "LinearTrend":
        model = LinearRegression().fit(x_train, y_train)
        return model, {}
    if name == "RandomForest":
        base_model = RandomForestRegressor(random_state=42, n_jobs=-1)
        grid = {"n_estimators": [180, 260, 340], "max_depth": [4, 6, 8], "min_samples_leaf": [1, 2]}
        cv_splits = 2 if len(y_train) < 12 else 3
        cv = TimeSeriesSplit(n_splits=cv_splits)
        gs = GridSearchCV(base_model, grid, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1, refit=True)
        gs.fit(x_train, y_train)
        return gs.best_estimator_, gs.best_params_
    raise ValueError(f"Unsupported model: {name}")


def build_model_from_params(name: str, params: dict):
    if name == "NaiveLag1":
        return None
    if name == "LinearTrend":
        return LinearRegression()
    if name == "RandomForest":
        return RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    raise ValueError(f"Unsupported model type: {name}")


def micro_forecast(series: pd.DataFrame, horizon: int):
    # Build lagged feature matrix.
    s = series[["Year", "Data"]].sort_values("Year").copy()
    s["lag1"] = s["Data"].shift(1)
    s["lag2"] = s["Data"].shift(2)
    s["d1"] = s["Data"].diff().shift(1)
    s = s.dropna().reset_index(drop=True)

    if len(s) < 8:
        return (
            pd.DataFrame(columns=["Year", "Model", "Forecast_MW", "Lower80_MW", "Upper80_MW"]),
            pd.DataFrame(columns=["Model", "MAE", "RMSE", "BacktestPoints", "BestParams"]),
        )

    x = s[["Year", "lag1", "lag2", "d1"]].to_numpy(dtype=float)
    y = s["Data"].to_numpy(dtype=float)
    candidates = [
        "NaiveLag1",
        "DriftTrend_TS",
        "SES_TS",
        "Blend_Naive_Drift_SES",
        "LinearTrend",
        "Blend_Naive_Linear_RF",
        "RandomForest",
    ]

    rows_eval = []
    rows_fc = []
    min_train = max(6, min(12, len(s) - 2))

    tuned_params = {name: {} for name in candidates}
    init_x = x[:min_train]
    init_y = y[:min_train]
    _, tuned = fit_model("RandomForest", init_x, init_y)
    tuned_params["RandomForest"] = tuned

    # Walk-forward backtest: train on past only, predict next point.
    for name in candidates:
        y_true_bt = []
        y_pred_bt = []

        for t in range(min_train, len(s)):
            x_train = x[:t]
            y_train = y[:t]
            x_next = x[t:t + 1]
            y_next = y[t]

            if name == "NaiveLag1":
                pred_next = float(x_next[0, 1])
            elif name == "DriftTrend_TS":
                pred_next = drift_next(y_train)
            elif name == "SES_TS":
                alpha_bt = fit_ses_alpha(y_train)
                pred_next = ses_next(y_train, alpha_bt)
            elif name == "Blend_Naive_Drift_SES":
                alpha_bt = fit_ses_alpha(y_train)
                pred_next = (float(x_next[0, 1]) + drift_next(y_train) + ses_next(y_train, alpha_bt)) / 3.0
            elif name == "Blend_Naive_Linear_RF":
                blend_lin = LinearRegression().fit(x_train, y_train)
                pred_lin = float(blend_lin.predict(x_next)[0])
                blend_rf = build_model_from_params("RandomForest", tuned_params.get("RandomForest", {}))
                blend_rf.fit(x_train, y_train)
                pred_rf = float(blend_rf.predict(x_next)[0])
                pred_next = (float(x_next[0, 1]) + pred_lin + pred_rf) / 3.0
            else:
                model_bt = build_model_from_params(name, tuned_params.get(name, {}))
                model_bt.fit(x_train, y_train)
                pred_next = float(model_bt.predict(x_next)[0])

            y_true_bt.append(float(y_next))
            y_pred_bt.append(max(0.0, pred_next))

        y_true_arr = np.array(y_true_bt, dtype=float)
        y_pred_arr = np.array(y_pred_bt, dtype=float)
        mae = float(np.mean(np.abs(y_true_arr - y_pred_arr))) if len(y_true_arr) else np.nan
        rmse = float(np.sqrt(np.mean((y_true_arr - y_pred_arr) ** 2))) if len(y_true_arr) else np.nan
        resid = float(np.std(y_true_arr - y_pred_arr)) if len(y_true_arr) > 1 else max(1.0, mae)

        if name == "NaiveLag1":
            model = None
            best_params = {}
        elif name == "DriftTrend_TS":
            model = None
            best_params = {"trend": "linear drift from first to last"}
        elif name == "SES_TS":
            alpha_full = fit_ses_alpha(y)
            model = {"alpha": alpha_full}
            best_params = {"alpha": alpha_full}
        elif name == "Blend_Naive_Drift_SES":
            alpha_full = fit_ses_alpha(y)
            model = {"alpha": alpha_full}
            best_params = {"blend": "(naive + drift + ses) / 3", "alpha": alpha_full}
        elif name == "Blend_Naive_Linear_RF":
            model = {
                "linear": LinearRegression().fit(x, y),
                "rf": build_model_from_params("RandomForest", tuned_params.get("RandomForest", {})),
            }
            model["rf"].fit(x, y)
            best_params = {
                "blend": "(naive + linear + random_forest) / 3",
                "rf_params": tuned_params.get("RandomForest", {}),
            }
        else:
            best_params = tuned_params.get(name, {})
            model = build_model_from_params(name, best_params)
            model.fit(x, y)

        rows_eval.append({
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "BacktestPoints": int(len(y_true_arr)),
            "BestParams": str(best_params),
        })

        vals = series["Data"].astype(float).tolist()
        last_year = int(series["Year"].max())
        for h in range(1, horizon + 1):
            y_next = last_year + h
            lag1 = vals[-1]
            lag2 = vals[-2] if len(vals) > 1 else vals[-1]
            d1 = vals[-1] - vals[-2] if len(vals) > 1 else 0.0

            if name == "NaiveLag1":
                y_hat = float(max(0.0, lag1))
            elif name == "DriftTrend_TS":
                drift = (float(vals[-1]) - float(vals[0])) / max(1, len(vals) - 1) if len(vals) >= 2 else 0.0
                y_hat = float(max(0.0, vals[-1] + drift))
            elif name == "SES_TS":
                alpha = float(model["alpha"])
                level = float(vals[0])
                for vi in vals[1:]:
                    level = alpha * float(vi) + (1.0 - alpha) * level
                y_hat = float(max(0.0, level))
            elif name == "Blend_Naive_Drift_SES":
                alpha = float(model["alpha"])
                level = float(vals[0])
                for vi in vals[1:]:
                    level = alpha * float(vi) + (1.0 - alpha) * level
                drift_part = float(vals[-1] + (vals[-1] - vals[0]) / max(1, len(vals) - 1)) if len(vals) >= 2 else float(vals[-1])
                y_hat = float(max(0.0, (float(lag1) + drift_part + level) / 3.0))
            elif name == "Blend_Naive_Linear_RF":
                x_next = np.array([[float(y_next), float(lag1), float(lag2), float(d1)]], dtype=float)
                y_lin = float(model["linear"].predict(x_next)[0])
                y_rf = float(model["rf"].predict(x_next)[0])
                y_hat = float(max(0.0, (lag1 + y_lin + y_rf) / 3.0))
            else:
                x_next = np.array([[float(y_next), float(lag1), float(lag2), float(d1)]], dtype=float)
                y_hat = float(max(0.0, model.predict(x_next)[0]))

            band = 1.28 * resid * (1.0 + 0.08 * (h - 1))
            rows_fc.append({
                "Year": y_next,
                "Model": name,
                "Forecast_MW": y_hat,
                "Lower80_MW": max(0.0, y_hat - band),
                "Upper80_MW": y_hat + band,
            })
            vals.append(y_hat)

    met = pd.DataFrame(rows_eval)
    met = met.replace([np.inf, -np.inf], np.nan).dropna(subset=["MAE", "RMSE"]).reset_index(drop=True)
    return pd.DataFrame(rows_fc), met

## Modelling Approach
The forecasting workflow uses rolling walk-forward evaluation: each model is trained on past years only and tested on the next unseen year. That keeps the exercise consistent with real forecasting conditions and avoids using future information during validation.

The feature set is intentionally compact.
- `lag1` and `lag2` capture persistence and short-memory behaviour.
- `d1` captures the most recent change in level so the models can respond to acceleration or slowdown without overcomplicating the design.

The candidate set combines baseline, smoothing, drift, linear, nonlinear, and blended models so that final selection is driven by observed series behaviour rather than by a single modelling assumption.

To keep the worked example aligned with the regional focus of the analysis, the detailed forecast illustration below uses Ontario wind rather than the Canada aggregate.

In [12]:
# EDA block used to justify modeling choices for the selected scope.
region = "ON"
source = "Wind"
horizon = 6

series = base[(base["Region"] == region) & (base["Source"] == source)][["Year", "Data"]].sort_values("Year")
if len(series) < 10:
    raise ValueError("Selected series has too few observations for reliable walk-forward evaluation.")

# 1) Trend evidence
fig_trend = px.line(series, x="Year", y="Data", markers=True, title=f"Historical Capacity Trend: {region} - {source}")
fig_trend.update_layout(template="plotly_white", yaxis_title="Capacity (MW)")
fig_trend.show()

# 2) Volatility evidence from region-source series (Canada aggregate excluded).
vol = region_only.groupby(["Region", "Source"], as_index=False).agg(Mean=("Data", "mean"), Std=("Data", "std"))
vol["Std"] = vol["Std"].fillna(0.0)
vol["CV_pct"] = np.where(vol["Mean"] > 0, 100.0 * vol["Std"] / vol["Mean"], 0.0)
fig_vol = px.scatter(
    vol,
    x="CV_pct",
    y="Mean",
    color="Source",
    hover_name="Region",
    title="Volatility vs Mean Capacity (Regional series only)",
    labels={"CV_pct": "Volatility (CV %)", "Mean": "Mean Capacity (MW)"},
)
fig_vol.update_layout(template="plotly_white")
fig_vol.show()

# 3) Walk-forward model evaluation for the selected series
fc, met = micro_forecast(series, horizon=horizon)
met = met.sort_values(["RMSE", "MAE"]).reset_index(drop=True)

best = met.iloc[0].to_dict()
best_model = str(best["Model"])

best_path = fc[fc["Model"] == best_model].sort_values("Year").reset_index(drop=True)

print(f"Region: {region}")
print(f"Source: {source}")
print(f"Horizon: {horizon} years")
print(f"Best model by walk-forward RMSE/MAE: {best_model}")
print(f"RMSE={float(best['RMSE']):.3f} | MAE={float(best['MAE']):.3f}")

display(met[["Model", "MAE", "RMSE", "BacktestPoints"]].round(3))

Region: ON
Source: Wind
Horizon: 6 years
Best model by walk-forward RMSE/MAE: DriftTrend_TS
RMSE=322.387 | MAE=236.748


,Model,MAE,RMSE,BacktestPoints
0,DriftTrend_TS,236.748,322.387,2
1,Blend_Naive_Drift_SES,596.679,635.081,2
2,LinearTrend,614.482,665.222,2
3,NaiveLag1,675.725,707.095,2
4,Blend_Naive_Linear_RF,684.346,771.513,2
5,SES_TS,895.488,919.203,2
6,RandomForest,1122.497,1149.125,2


In [13]:
# Forecast deliverable visualization (historical + best-model future + interval).
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=series["Year"],
    y=series["Data"],
    mode="lines+markers",
    name="Historical",
    line={"width": 3, "color": "#0f172a"},
))
fig.add_trace(go.Scatter(
    x=best_path["Year"],
    y=best_path["Forecast_MW"],
    mode="lines+markers",
    name=f"Forecast ({best_model})",
    line={"width": 3, "color": "#1d4ed8"},
))
fig.add_trace(go.Scatter(
    x=best_path["Year"],
    y=best_path["Lower80_MW"],
    mode="lines",
    name="Lower80",
    line={"width": 1, "dash": "dash", "color": "#6b7280"},
))
fig.add_trace(go.Scatter(
    x=best_path["Year"],
    y=best_path["Upper80_MW"],
    mode="lines",
    name="Upper80",
    line={"width": 1, "dash": "dash", "color": "#6b7280"},
))
fig.update_layout(
    template="plotly_white",
    title="Forecast Deliverable (Best model from walk-forward ranking)",
    xaxis_title="Year",
    yaxis_title="Capacity (MW)",
    legend_title="Series",
    height=460,
)
fig.show()

display(best_path[["Year", "Forecast_MW", "Lower80_MW", "Upper80_MW"]].round(2))

,Year,Forecast_MW,Lower80_MW,Upper80_MW
0,2017,5280.22,4977.18,5583.26
1,2018,5718.99,5391.71,6046.27
2,2019,6157.75,5806.23,6509.28
3,2020,6596.52,6220.76,6972.29
4,2021,7035.29,6635.28,7435.30
5,2022,7474.06,7049.81,7898.31


In [14]:
# Findings, conclusions, and recommendations grounded in computed outputs.
top2 = met[["Model", "RMSE", "MAE"]].head(2).copy()
runner_up = top2.iloc[1]["Model"] if len(top2) > 1 else "n/a"
rmse_gap = float(top2.iloc[1]["RMSE"] - top2.iloc[0]["RMSE"]) if len(top2) > 1 else 0.0

series_arr = series["Data"].to_numpy(dtype=float)
lag_corr = float(np.corrcoef(series_arr[1:], series_arr[:-1])[0, 1]) if len(series_arr) >= 3 else np.nan
series_cv = float(np.std(series_arr) / np.mean(series_arr)) if np.mean(series_arr) > 0 else np.nan

summary = pd.DataFrame([
    {
        "Finding": "Model ranking",
        "Evidence": f"Best model is {best_model}; runner-up is {runner_up}; RMSE gap = {rmse_gap:.3f}.",
        "Conclusion": "Model choice should be series-specific and performance-led.",
        "Recommendation": "Use per-series walk-forward ranking as the default selection rule.",
    },
    {
        "Finding": "Series behavior",
        "Evidence": f"Lag-1 correlation = {lag_corr:.3f}; coefficient of variation = {series_cv:.3f}.",
        "Conclusion": "Persistence and variability are both material, supporting a mixed model portfolio.",
        "Recommendation": "Retain baseline, smoothing, trend, nonlinear, and blend candidates in evaluation.",
    },
    {
        "Finding": "Forecast usability",
        "Evidence": f"Forecast horizon {int(best_path['Year'].min())}-{int(best_path['Year'].max())} includes point path and 80% interval.",
        "Conclusion": "Outputs are suitable for planning discussion because uncertainty is explicit.",
        "Recommendation": "Report both central path and interval bounds in decision briefings.",
    },
])

model_catalog = pd.DataFrame([
    {"Model": "NaiveLag1", "Family": "Baseline persistence"},
    {"Model": "DriftTrend_TS", "Family": "Time-series drift"},
    {"Model": "SES_TS", "Family": "Time-series smoothing"},
    {"Model": "Blend_Naive_Drift_SES", "Family": "Time-series ensemble"},
    {"Model": "LinearTrend", "Family": "Linear regression"},
    {"Model": "Blend_Naive_Linear_RF", "Family": "Hybrid ensemble"},
    {"Model": "RandomForest", "Family": "Tree ensemble"},
])

display(summary)
display(model_catalog)

,Finding,Evidence,Conclusion,Recommendation
0,Model ranking,Best model is DriftTrend_TS; runner-up is Blen...,Model choice should be series-specific and per...,Use per-series walk-forward ranking as the def...
1,Series behavior,Lag-1 correlation = 0.987; coefficient of vari...,"Persistence and variability are both material,...","Retain baseline, smoothing, trend, nonlinear, ..."
2,Forecast usability,Forecast horizon 2017-2022 includes point path...,Outputs are suitable for planning discussion b...,Report both central path and interval bounds i...


,Model,Family
0,NaiveLag1,Baseline persistence
1,DriftTrend_TS,Time-series drift
2,SES_TS,Time-series smoothing
3,Blend_Naive_Drift_SES,Time-series ensemble
4,LinearTrend,Linear regression
5,Blend_Naive_Linear_RF,Hybrid ensemble
6,RandomForest,Tree ensemble


## References

Canada Energy Regulator. (n.d.). *Electricity capacity dataset* [CSV file]. https://www.cer-rec.gc.ca/open/energy/electricity-capacity-dataset.csv

Government of Canada, Open Government Portal. (n.d.). *Electricity generation and capacity in Canada - Electricity capacity* [Dataset resource]. https://open.canada.ca/data/en/dataset/2cdf43fc-d4aa-4604-9f21-29777d955810/resource/2971c342-94e7-416b-8e61-02c64c532708

Government of Canada, Open Government Portal. (n.d.). *Electricity capacity - Guide* [Data dictionary]. https://open.canada.ca/data/en/dataset/2cdf43fc-d4aa-4604-9f21-29777d955810/resource/3ebcbfdb-4c04-4293-8a02-a8da19067084

Hyndman, R. J., & Athanasopoulos, G. (n.d.). *Forecasting: Principles and practice* (3rd ed.). OTexts. https://otexts.com/fpp3/

NumPy Developers. (n.d.). *NumPy documentation*. https://numpy.org/doc/

pandas development team. (n.d.). *pandas documentation*. https://pandas.pydata.org/docs/

Plotly Technologies Inc. (n.d.). *Plotly Python documentation*. https://plotly.com/python/

scikit-learn Developers. (n.d.). *scikit-learn user guide*. https://scikit-learn.org/stable/user_guide.html

Note. Retrieval date for all web resources: March 30, 2026.